# 🚀 Speech Challenge 2026 - 구글 코랩(Colab) 원클릭 실행 템플릿

이 노트북은 **Speech Challenge 2026** (극한 전장 환경 음성 인식 모델 개발) 프로젝트를 Google Colab 환경에서 손쉽게 가져와 학습 및 평가할 수 있도록 구성된 전용 템플릿입니다.

### 📌 실행 전 체크리스트
1. **GPU 런타임 활성화**: 상단 메뉴 `런타임` ➔ `런타임 유형 변경` ➔ 하드웨어 가속기를 **GPU (T4, V100, A100 등)**로 선택합니다.
2. **순차적 실행**: 위에서부터 셀을 하나씩 순서대로 `Shift + Enter`로 실행합니다.

## 1. 구글 드라이브 마운트 (Google Drive Mount)
대용량 학습 데이터셋(`.parquet`)을 불러오고, 학습된 모델 가중치(체크포인트)를 세션 종료 후에도 영구 보존하기 위해 구글 드라이브를 연결합니다.

In [1]:
# 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

## 2. 깃허브 리포지토리 클론 및 작업 디렉토리 이동
본인의 GitHub 리포지토리 코드를 Colab 인스턴스로 복제(`git clone`)하고, 프로젝트 루트 폴더로 진입합니다.

> 💡 **주의**: 아래 URL의 `[본인_깃허브_주소]` 부분을 본인의 GitHub 리포지토리 주소(예: `https://github.com/username/Speech-Challenge-2026.git`)로 수정해 주세요.

In [ ]:
# 깃허브 리포지토리 클론
# 예시: !git clone https://github.com/jyj-08/Speech-Challenge-2026.git
!git clone [jyj-08]

# 프로젝트 디렉토리 내부로 이동
%cd Speech-Challenge-2026

# 현재 작업 경로 및 파일 확인
!pwd
!ls -la

## 3. GPU 환경 확인 및 패키지 의존성 설치
할당된 GPU의 상태를 점검하고, 프로젝트 구동에 필요한 패키지(`torch`, `transformers`, `peft`, `audiomentations` 등)를 `requirements.txt`를 통해 한 번에 설치합니다.

In [ ]:
# 1) GPU 할당 및 VRAM 상태 확인
!nvidia-smi

# 2) requirements.txt 패키지 일괄 설치
!pip install --upgrade pip
!pip install -r requirements.txt

## 4. 데이터셋 준비 (Google Drive ➔ Colab 로컬 경로 복사)
Colab 인스턴스 내부에 `data/raw/` 디렉토리를 생성하고, 구글 드라이브에 업로드되어 있는 `.parquet` 데이터 파일들을 로컬 SSD 공간으로 복사해 옵니다.

> 💡 **안내**: `/content/drive/MyDrive/[데이터셋_경로]/` 부분을 실제 구글 드라이브 내 parquet 파일이 위치한 폴더 경로로 변경해 주세요.

In [ ]:
# 1) 로컬 데이터 디렉토리 생성
!mkdir -p data/raw

# 2) 구글 드라이브에서 .parquet 데이터 파일 복사
# 예시: !cp /content/drive/MyDrive/SpeechData/*.parquet data/raw/
!cp /content/drive/MyDrive/[데이터셋_경로]/*.parquet data/raw/

# 3) 데이터 복사 완료 확인 (파일 목록 및 용량)
!ls -lh data/raw/

## 5. 드라이 런 (Dry-Run: 파이프라인 빠른 사전 테스트)
본격적인 대규모 학습에 앞서, **GPU Out-Of-Memory(OOM) 에러**나 **데이터 전처리/모델 텐서 차원 버그**가 없는지 5 step만 초경량으로 실행하여 정상 구동을 검증합니다.

* `--epochs 1`: 1 에포크로 제한
* `--batch_size 2`: 적은 배치 사이즈로 OOM 방지 및 빠른 초기화
* `--max_steps 5`: 5 스텝만 실행 후 즉시 종료하여 파이프라인 정상 동작 여부만 검증

In [ ]:
# 드라이 런 실행 (로컬 parquet 데이터가 있다면 --data_files 'data/raw/*.parquet' 추가 가능)
!python src/train.py \
    --config configs/default_config.yaml \
    --epochs 1 \
    --batch_size 2 \
    --max_steps 5 \
    --output_dir ./checkpoints/dry_run_test

## 6. 본 학습 (Main Training) 및 체크포인트 구글 드라이브 자동 백업
드라이 런 검증이 완료되면 전체 데이터를 대상으로 본격적인 LoRA 파인튜닝을 진행합니다.

체크포인트 저장 경로(`--output_dir`)를 **마운트된 구글 드라이브 경로**(`/content/drive/MyDrive/...`)로 직접 지정하여, 코랩 세션이 끊기거나 타임아웃이 발생해도 학습된 모델 가중치가 안전하게 영구 저장되도록 설정합니다.

In [ ]:
# 1) 구글 드라이브 내 체크포인트 저장 폴더 생성
!mkdir -p /content/drive/MyDrive/Speech-Challenge-2026/checkpoints/tactical_whisper

# 2) 본 학습 파이프라인 실행
!python src/train.py \
    --config configs/default_config.yaml \
    --epochs 3 \
    --batch_size 4 \
    --learning_rate 0.001 \
    --output_dir /content/drive/MyDrive/Speech-Challenge-2026/checkpoints/tactical_whisper

# 3) 구글 드라이브에 저장된 모델 가중치 파일 확인
!ls -lh /content/drive/MyDrive/Speech-Challenge-2026/checkpoints/tactical_whisper

## 7. (선택) 학습 완료 모델 평가 (Evaluation)
구글 드라이브에 저장된 LoRA 체크포인트를 불러와 WER 및 CER 성능을 평가하고 결과를 JSON 리포트로 저장합니다.

In [ ]:
!python src/evaluate.py \
    --checkpoint_dir /content/drive/MyDrive/Speech-Challenge-2026/checkpoints/tactical_whisper \
    --metrics wer cer \
    --output_file /content/drive/MyDrive/Speech-Challenge-2026/checkpoints/eval_results.json